In [0]:
# =============================================================================
# ICEBASE — PHASE 3 | NOTEBOOK 03
# Gold Layer — retention_cohort
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# STANDALONE NOTEBOOK — Run attached to icebase-dev cluster.
# Reads from Silver, writes to icebase.gold.retention_cohort.
#
# WHAT THIS NOTEBOOK DOES:
#   Builds retention cohort data — the training ground for the churn
#   prediction model in Phase 4. For each customer, it calculates:
#     - Did they return within 30 days of their first purchase?
#     - Did they return within 60 days?
#     - Are they "churned" (no purchase in 45+ days)?
#
#   This table answers the CMO's hardest question:
#   "Which fans did we lose and when — and can we predict it before it happens?"
#
#   It also powers the "bad story" analysis:
#   Walk-up buyers from the hot start who never returned are visible here.
#   Jersey Night cohort 30-day return rate is measurable here.
#
# TABLE WRITTEN:
#   icebase.gold.retention_cohort
# =============================================================================

In [0]:
# COMMAND ----------
# ── CELL 1: Imports and Config ─────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
CATALOG = "icebase"
GOLD    = f"{CATALOG}.gold"
SILVER  = f"{CATALOG}.silver"
 
print("✅ retention_cohort notebook loaded")
print(f"   Writing to: {GOLD}.retention_cohort")

In [0]:
# COMMAND ----------
# ── CELL 2: Build per-customer purchase timeline ───────────────────────────
#
# For each customer, find:
#   first_purchase_date : when they first bought a ticket
#   second_purchase_date: when (if ever) they bought again
#   last_purchase_date  : most recent purchase
#   first_game_phase    : which season phase they entered during
 
fact  = spark.table(f"{SILVER}.fact_tickets")
dim_g = spark.table(f"{SILVER}.dim_game")
dim_c = spark.table(f"{SILVER}.dim_customer")
 
# Enrich tickets with game date and phase
fact_with_phase = (
    fact.alias("f")
    .join(
        dim_g.select("game_id", "game_date", "season_phase", "is_jersey_night").alias("g"),
        "game_id", "left"
    )
)
 
# Window: order each customer's purchases chronologically
w_cust = Window.partitionBy("customer_id").orderBy("g.game_date")
 
# Per-customer purchase sequence
purchase_seq = (
    fact_with_phase
    .withColumn("purchase_rank", F.rank().over(w_cust))
    .groupBy("customer_id")
    .agg(
        F.min("g.game_date").cast("date")             .alias("first_purchase_date"),
        F.max("g.game_date").cast("date")             .alias("last_purchase_date"),
        F.countDistinct("game_id")                    .alias("total_games"),
        F.round(F.sum("ticket_price"), 2)             .alias("total_spend"),
 
        # First game's season phase = acquisition context
        F.first("g.season_phase", ignorenulls=True)   .alias("acquisition_phase"),
 
        # Did they attend Jersey Night?
        F.max(F.col("g.is_jersey_night").cast("int")).cast("boolean")
                                                       .alias("attended_jersey_night"),
 
        # Was any purchase a promo?
        F.max(F.col("is_promo_ticket").cast("int")).cast("boolean")
                                                       .alias("ever_used_promo"),
 
        # Promo rate across all purchases
        F.round(
            F.sum(F.when(F.col("is_promo_ticket"), 1).otherwise(0))
            / F.count("ticket_id"), 4
        )                                              .alias("overall_promo_rate"),
    )
)

In [0]:
# COMMAND ----------
# ── CELL 3: Calculate retention windows ───────────────────────────────
#
# For each fan, measure whether they returned within key time windows.
# This creates the ground truth for the churn model:
#   returned_30d : bought again within 30 days of first purchase
#   returned_60d : bought again within 60 days of first purchase
#   churn_flag   : no purchase in 45+ days from last purchase (churned)
#   days_to_second_purchase: how long before they came back (loyalty signal)
 
# Get the date of the second purchase per customer
second_purchase = (
    fact_with_phase
    .withColumn("purchase_rank", F.dense_rank().over(
        Window.partitionBy("customer_id").orderBy("g.game_date")
    ))
    .filter(F.col("purchase_rank") == 2)
    .groupBy("customer_id")
    .agg(F.min("g.game_date").cast("date").alias("second_purchase_date"))
)
 
retention_cohort = (
    purchase_seq
    .join(second_purchase, "customer_id", "left")
    .join(
        dim_c.select(
            "customer_id", "acquisition_channel", "initial_segment",
            "is_season_holder", "tenure_days", "is_jersey_night_acq"
        ),
        "customer_id", "left"
    )
    # Days between first and second purchase
    .withColumn("days_to_second_purchase",
        F.datediff(
            F.col("second_purchase_date"),
            F.col("first_purchase_date")
        )
    )
    # Return windows
    .withColumn("returned_30d",
        F.when(F.col("days_to_second_purchase") <= 30, True).otherwise(False)
    )
    .withColumn("returned_60d",
        F.when(F.col("days_to_second_purchase") <= 60, True).otherwise(False)
    )
    # Churn flag: last purchase was 45+ days ago
    .withColumn("churn_flag",
        F.when(
            F.datediff(F.current_date(), F.col("last_purchase_date")) > 45,
            True
        ).otherwise(False)
    )
    # Days since last purchase (recency for ML)
    .withColumn("days_since_last",
        F.datediff(F.current_date(), F.col("last_purchase_date"))
    )
    # One-and-done flag: only attended one game ever
    .withColumn("is_one_and_done",
        F.when(F.col("total_games") == 1, True).otherwise(False)
    )
    # High value flag: total spend in top 20% (rough threshold)
    .withColumn("is_high_value",
        F.when(F.col("total_spend") >= 300, True).otherwise(False)
    )
    # Jersey Night cohort flag — key business story
    .withColumn("is_jersey_night_cohort",
        F.col("attended_jersey_night") | F.col("is_jersey_night_acq")
    )
    .withColumn("_gold_refreshed_at", F.current_timestamp())
)

In [0]:
# COMMAND ----------
# ── CELL 4: Write to Gold ──────────────────────────────────────────────────
 
(
    retention_cohort
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.retention_cohort")
)
 
count = spark.table(f"{GOLD}.retention_cohort").count()
print(f"✅ retention_cohort written | {count:,} rows")

In [0]:
# COMMAND ----------
# ── CELL 5: Quick Sanity Checks ────────────────────────────────────────────
 
print("── Churn rate by acquisition phase ──")
display(
    spark.table(f"{GOLD}.retention_cohort")
    .groupBy("acquisition_phase")
    .agg(
        F.count("*")                                        .alias("fans"),
        F.sum(F.col("churn_flag").cast("int"))              .alias("churned"),
        F.round(
            F.sum(F.col("churn_flag").cast("int"))
            / F.count("*") * 100, 1
        )                                                   .alias("churn_rate_pct"),
        F.round(F.avg("days_since_last"), 0)               .alias("avg_days_since_last"),
        F.sum(F.col("returned_30d").cast("int"))            .alias("returned_30d"),
        F.round(
            F.sum(F.col("returned_30d").cast("int"))
            / F.count("*") * 100, 1
        )                                                   .alias("return_30d_rate_pct"),
    )
    .orderBy(F.col("churn_rate_pct").desc())
)
 
# EXPECTED KEY PATTERNS:
#   walk-up / social_media acquisition → highest churn_rate_pct
#   season_package acquisition → lowest churn_rate_pct
#   jersey_night cohort → high return_30d_rate_pct (the good story)
 
print("\n── Jersey Night cohort 30-day return analysis ──")
display(
    spark.table(f"{GOLD}.retention_cohort")
    .groupBy("is_jersey_night_cohort")
    .agg(
        F.count("*")                                        .alias("fans"),
        F.round(
            F.sum(F.col("returned_30d").cast("int"))
            / F.count("*") * 100, 1
        )                                                   .alias("return_30d_rate_pct"),
        F.round(F.avg("days_since_last"), 0)               .alias("avg_days_since_last"),
    )
)
# EXPECTED: is_jersey_night_cohort=TRUE has significantly higher return_30d_rate_pct
# This is the "good story" — Jersey Night reactivated and converted fans